# Konsolidierung der EMR-Ist-Datenbasis

Dieses Notebook baut aus den EMR-Rohdaten der SPEDION-Fahrer-App eine bereinigte Ist-Basis und speichert sie als CSV und Pickle. Wie beim Soll folgt es einem gemeinsamen Rückgrat: Einlesen, auf März filtern, Mafi entfernen, Fahrer und Kennzeichen normalisieren, Dubletten entfernen, exportieren. Anders als das Soll liegen die EMR-Daten auf Meldungsebene vor, also als einzelne GPS- und Statusmeldungen ohne Tour-Struktur. Eine Tour-Zuordnung entsteht erst später im Analyse-Notebook; hier geht es darum, die Meldungen sauber und über das Kennzeichen anschlussfähig zu machen.

## EMR-Rohdaten einlesen und konsolidieren

Die Ist-Daten liegen als fünf Einzelmeldungsreports vor, je eine Datei pro Kalenderwoche (02.03. bis 05.04.2026). Jede Datei enthält ein Zusammenfassungs-Sheet sowie je ein Detail-Sheet pro Fahrzeug. Das Zusammenfassungs-Sheet überspringen wir bewusst: Es enthält nur eine aggregierte Wochenübersicht und keine einzelnen Meldungen, würde unsere zeilenweise Meldungsbasis also mit Aggregatzeilen verfälschen. Wir greifen deshalb gezielt nur die Detail-Sheets ab, erkennbar an der Endung "- Fahrzeugdetails" im Sheet-Namen. In den Detail-Sheets beginnen die Meldungen ab Zeile 9, die Kopfzeile steht in Zeile 8. Das Kennzeichen steht nur im Sheet-Namen, daher übernehmen wir es als eigene Spalte QUELLE_KENNZEICHEN. Alle Detail-Sheets aller fünf Wochen hängen wir zu einem durchgehenden DataFrame emr_raw zusammen.

In [1]:
import pandas as pd
from pathlib import Path
import re

# Anzeigeoptionen, damit DataFrames vollständig sichtbar sind
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 60)
pd.set_option("display.width", None)

BASE_PATH = Path.cwd().parent / "data" / "raw"

# Alle EMR-Wochendateien (je Datei = eine Kalenderwoche)
emr_dateien = sorted(BASE_PATH.glob("Einzelmeldungsreport_*.xlsx"))

teile = []
for datei in emr_dateien:
    xl = pd.ExcelFile(datei)
    detail_sheets = [s for s in xl.sheet_names if s.endswith("- Fahrzeugdetails")]
    for sheet in detail_sheets:
        teil = pd.read_excel(datei, sheet_name=sheet, header=8)  # Kopfzeile in Zeile 8
        teil["QUELLE_KENNZEICHEN"] = sheet.replace(" - Fahrzeugdetails", "")
        teil["QUELLE_DATEI"] = datei.name
        teile.append(teil)

emr_raw = pd.concat(teile, ignore_index=True)
print(f"Dateien: {len(emr_dateien)} | Zeilen: {len(emr_raw)}")
emr_raw.head()

Dateien: 5 | Zeilen: 188720


,Fahrer PIN,Fahrername,Formularnummer,Formularbeschreibung,Meldungszeit,Richtung,Geschwindigkeit,KM-Stand,Gesamtverbrauch,Tankfüllstand,Breitengrad,Längengrad,Position,Meldungsquelle,QUELLE_KENNZEICHEN,QUELLE_DATEI
0,NaN,,1332,Tour im Fahrzeug angekommen,2026-03-02 12:52:19,180,1,0.0,0.0,0.0,"53,52311","9,98774","DE - 20457 Hamburg (Hamburg-Mitte), Brandenburger Straße 12",SPEDION App,Mafi,Einzelmeldungsreport_HTS-Cargo_GmbH_2026_04_14-12_27_05....
1,NaN,,1382,TourDeleteByID - OK,2026-03-02 12:52:20,0,0,0.0,0.0,0.0,"53,52297","9,98831","DE - 20457 Hamburg (Hamburg-Mitte), Brandenburger Straße 17",SPEDION App,Mafi,Einzelmeldungsreport_HTS-Cargo_GmbH_2026_04_14-12_27_05....
2,NaN,,1344,"Tour, Place oder Order im Fahrzeug gelöscht",2026-03-02 12:52:20,0,0,0.0,0.0,0.0,"53,52297","9,98831","DE - 20457 Hamburg (Hamburg-Mitte), Brandenburger Straße 17",SPEDION App,Mafi,Einzelmeldungsreport_HTS-Cargo_GmbH_2026_04_14-12_27_05....
3,NaN,,1332,Tour im Fahrzeug angekommen,2026-03-02 12:52:21,0,0,0.0,0.0,0.0,"53,52298","9,98828","DE - 20457 Hamburg (Hamburg-Mitte), Brandenburger Straße 17",SPEDION App,Mafi,Einzelmeldungsreport_HTS-Cargo_GmbH_2026_04_14-12_27_05....
4,NaN,,2100,SPEDION Mobile Control Nutzung,2026-03-02 12:52:25,0,0,0.0,0.0,0.0,"53,523","9,98832","DE - 20457 Hamburg (Hamburg-Mitte), Brandenburger Straße 17",SPEDION App,Mafi,Einzelmeldungsreport_HTS-Cargo_GmbH_2026_04_14-12_27_05....


## Untersuchung und Aufbereitung der EMR-Daten

Wir starten mit info(), um Datentypen und Befüllung der 16 Spalten zu sehen. Daraus ergeben sich direkt die ersten Schritte: Breitengrad und Längengrad liegen als Text vor und müssen in Zahlen umgewandelt werden, und Fahrer PIN ist nur teilweise gefüllt - die Fahrer-Spalten sehen wir uns deshalb gleich genauer an.

In [2]:
emr_raw.info()

<class 'pandas.DataFrame'>
RangeIndex: 188720 entries, 0 to 188719
Data columns (total 16 columns):
 #   Column                Non-Null Count   Dtype         
---  ------                --------------   -----         
 0   Fahrer PIN            165978 non-null  float64       
 1   Fahrername            188720 non-null  str           
 2   Formularnummer        188720 non-null  int64         
 3   Formularbeschreibung  188720 non-null  str           
 4   Meldungszeit          188720 non-null  datetime64[us]
 5   Richtung              188720 non-null  int64         
 6   Geschwindigkeit       188720 non-null  int64         
 7   KM-Stand              188720 non-null  float64       
 8   Gesamtverbrauch       188720 non-null  float64       
 9   Tankfüllstand         188720 non-null  float64       
 10  Breitengrad           188720 non-null  str           
 11  Längengrad            188720 non-null  str           
 12  Position              188720 non-null  str           
 13  Meldungsqu

## Koordinaten in Zahlen umwandeln

Breitengrad und Längengrad kommen als Komma-Text und werden zu float, damit sie für die spätere Geofence- und Routenanalyse nutzbar sind. Das describe() am Ende dient als Kontrolle: Der Wertebereich passt zu Norddeutschland, und anders als bei den Soll-Daten gibt es hier keine Null-Koordinaten.

In [3]:
# Breitengrad/Längengrad liegen als Komma-Text vor -> in float umwandeln
for spalte in ["Breitengrad", "Längengrad"]:
    emr_raw[spalte] = pd.to_numeric(emr_raw[spalte].str.replace(",", ".", regex=False))

emr_raw[["Breitengrad", "Längengrad"]].describe()

,Breitengrad,Längengrad
count,188720.000000,188720.000000
mean,53.547536,9.966510
std,0.203165,0.361147
min,51.298260,8.152620
25%,53.520570,9.964250
50%,53.523230,9.986065
75%,53.585160,9.989430
max,54.781510,13.649580


## Auf den Analysezeitraum März beschränken

Das describe() im Schritt davor hat gezeigt, dass die EMR-Wochen bis zum 5. April reichen. Wie beim Soll begrenzen wir auf März 2026 mit denselben Grenzen (>= 2026-03-01 und < 2026-04-01), damit alle drei Quellen denselben Zeitraum abdecken. Wir filtern über die Meldungszeit. Es fallen die April-Zeilen weg, der Rest bleibt vollständig.

In [4]:
vorher = len(emr_raw)
maerz = (emr_raw["Meldungszeit"] >= "2026-03-01") & (emr_raw["Meldungszeit"] < "2026-04-01")
emr_raw = emr_raw[maerz].copy()
print(f"Außerhalb März entfernt: {vorher - len(emr_raw)} Zeilen | verbleibend: {len(emr_raw)}")

Außerhalb März entfernt: 16764 Zeilen | verbleibend: 171956


nunique() gibt einen Kardinalitäts-Überblick und trennt die hochkardinalen Spalten (Zeitstempel, Koordinaten, KM-Stand) von den wenigen kategorialen, die wir als Nächstes über ihre Verteilung ansehen.

In [5]:
emr_raw.nunique()

Fahrer PIN                  14
Fahrername                  15
Formularnummer              77
Formularbeschreibung        76
Meldungszeit            142463
Richtung                   361
Geschwindigkeit             99
KM-Stand                 48880
Gesamtverbrauch          14462
Tankfüllstand              247
Breitengrad              24394
Längengrad               31958
Position                  6877
Meldungsquelle               2
QUELLE_KENNZEICHEN          13
QUELLE_DATEI                 5
dtype: int64

Die Verteilung der kategorialen Spalten zeigt dreierlei: In QUELLE_KENNZEICHEN taucht das Fahrzeug Mafi auf, das im nächsten Schritt entfernt wird; Formularbeschreibung listet die Meldungstypen im Klartext; und Meldungsquelle hat nur zwei Werte (SPEDION App, TruckBox) und ist damit ein Kandidat für die abschließende Reduktion.

In [6]:
# Verteilung der kategorialen Spalten
for spalte in ["QUELLE_KENNZEICHEN", "Formularbeschreibung", "Meldungsquelle"]:
    display(emr_raw[spalte].value_counts(dropna=False))

QUELLE_KENNZEICHEN
Mafi          20989
PI-EN 444     17872
Plö-TS853     16794
TS859         16387
OD-TS 8041    14518
OD-TS 8046    13966
OD-TS 8050    13906
Plö-TS856     13321
WL-PL431      12143
OD-TS 8048    10469
OD-TS 8044     9187
PI-EN 900      7055
Plo-TS857      5349
Name: count, dtype: int64

Formularbeschreibung
Rollen                                               43173
Beginn Rollen                                        34188
Ende Rollen                                          34162
Anhalten                                             23515
Place wurde aktiviert                                10039
                                                     ...  
Tankfüllstand Event - Große Abnahme                      1
Support Info mit Anhang                                  1
Android Update installed by MDM successful               1
Warten                                                   1
Storno (Tour, Place oder Order) nicht verarbeitet        1
Name: count, Length: 76, dtype: int64

Meldungsquelle
SPEDION App    92711
TruckBox       79245
Name: count, dtype: int64

## Mafi entfernen

In QUELLE_KENNZEICHEN taucht das Fahrzeug Mafi auf, das wie bei den Soll-Daten nicht in die Analyse gehört. Wir entfernen alle Mafi-Zeilen, 20.989 an der Zahl. Das Mafi-Fahrzeug wird von zwei Fahrern gefahren: Mirzaie fällt damit komplett weg, weil er ausschließlich Mafi fuhr. Stoffer bleibt dagegen erhalten, denn neben seinen Mafi-Fahrten hat er elf Meldungen auf dem regulären Fahrzeug WL-PL431. Die Zahl der Fahrer sinkt damit von 14 auf 13.

In [7]:
vorher = len(emr_raw)
emr_raw = emr_raw[emr_raw["QUELLE_KENNZEICHEN"] != "Mafi"].copy()
print(f"Mafi entfernt: {vorher - len(emr_raw)} Zeilen | verbleibend: {len(emr_raw)}")

Mafi entfernt: 20989 Zeilen | verbleibend: 150967


Weil info() die Lücke in Fahrer PIN gezeigt hat, sehen wir uns beide Fahrer-Spalten über ihre Häufigkeiten an. Auffällig: Der häufigste Fahrername ist ein leerer Eintrag (Leerzeichen, kein NaN, weshalb info() ihn nicht als fehlend zählt), und Fahrer PIN hat genau gleich viele NaN. Beide Spalten tragen dieselbe Information mit derselben Lücke, einmal als Code, einmal als Klartext.

In [8]:
# Fahrer-Spalten über ihre Häufigkeiten ansehen
for spalte in ["Fahrername", "Fahrer PIN"]:
    display(emr_raw[spalte].value_counts(dropna=False).head(10))

Fahrername
                            20786
Krüger, Alexander           16021
Jorczik, Christian          14658
Hruszka, Krzysztof          14057
Calina, Florin Alexandru    13646
Hesse, Sven-Erik            13131
Giewon, Mariusz             12893
Szmajdewicz, Marcin         11823
Oelschläger, Marcel          9471
Borrs, Thomas                8436
Name: count, dtype: int64

Fahrer PIN
NaN       20786
8180.0    16021
8160.0    14658
8197.0    14057
8168.0    13646
8317.0    13131
8310.0    12893
8311.0    11823
8258.0     9471
8205.0     8436
Name: count, dtype: int64

## Fahrernamen angleichen und nicht benötigte Spalten entfernen

Zwei Erkenntnisse setzen wir hier um. Erstens bringen wir Fahrername auf den Stand der Soll-Daten: leere Einträge zu echtem NaN, Umlaute ausgeschrieben (Krüger zu Krueger, Oelschläger zu Oelschlaeger) und zwei abweichende Schreibweisen derselben Personen an die Soll-Schreibweise angeglichen (Hruszka, Krzysztof zu Hrzuska, Krzystof und Teme, Abdoulaeye zu Teme, Abdoulaye). Auch wenn die Soll-Form wie ein Tippfehler aussieht, ist sie unsere Referenz, an der sich beide Ist-Quellen ausrichten. Zweitens entfernen wir Spalten, deren fehlende Relevanz sich direkt erschließt: Fahrer PIN (redundant zum Namen), Gesamtverbrauch und Tankfüllstand (Kraftstoff, kein Bezug zum Projektziel) sowie Richtung (Kompasskurs, für die Tour-Rekonstruktion durch die Koordinaten bereits abgedeckt).

In [9]:
# Fahrername auf das Soll-Schema bringen
umlaut_expand = str.maketrans({"ä": "ae", "ö": "oe", "ü": "ue", "Ä": "Ae", "Ö": "Oe", "Ü": "Ue", "ß": "ss"})
namens_korrektur = {
    "Hruszka, Krzysztof": "Hrzuska, Krzystof",   # gleiche Person, abweichende Schreibweise
    "Teme, Abdoulaeye": "Teme, Abdoulaye",
}
emr_raw["Fahrername"] = (
    emr_raw["Fahrername"]
    .str.strip().replace("", pd.NA)    # Leerzeichen -> NaN
    .str.translate(umlaut_expand)      # Umlaute wie im Soll ausschreiben
    .replace(namens_korrektur)         # Schreibvarianten angleichen
)

emr_raw = emr_raw.drop(columns=["Fahrer PIN", "Gesamtverbrauch", "Tankfüllstand", "Richtung"])

print("Fahrername fehlend:", emr_raw["Fahrername"].isna().sum())
print("Spalten verbleibend:", emr_raw.shape[1])

Fahrername fehlend: 20786
Spalten verbleibend: 12


Ohne die Kraftstoff- und Richtungsspalten ist describe() jetzt aussagekräftig. Bei Geschwindigkeit und KM-Stand fällt jeweils ein hoher Null-Anteil auf (das 25-Prozent-Quartil liegt bei 0), den wir als Nächstes untersuchen. Formularnummer und die Koordinaten sind als Statistik ohne Aussage. Nebenbei zeigt describe() bei der Meldungszeit noch, dass die Rohdaten bis in den April reichten — der Anlass für den Zeitfilter, den wir bereits zuvor gesetzt haben.

In [10]:
emr_raw.describe()

,Formularnummer,Meldungszeit,Geschwindigkeit,KM-Stand,Breitengrad,Längengrad
count,150967.000000,150967,150967.000000,150967.000000,150967.000000,150967.000000
mean,16551.620076,2026-03-16 05:44:03.360780,4.230256,281596.023097,53.553546,9.961703
min,1156.000000,2026-03-02 00:00:00,0.000000,0.000000,51.298260,8.152620
25%,1643.000000,2026-03-09 11:38:19,0.000000,143998.877500,53.519240,9.956760
50%,30001.000000,2026-03-16 11:27:45,3.000000,196155.845000,53.523530,9.985270
75%,30019.000000,2026-03-24 07:24:28,5.000000,462368.745000,53.600120,9.992080
max,30039.000000,2026-03-31 23:56:00,116.000000,808523.005000,54.781510,13.649580
std,14107.458261,NaN,6.411984,225662.537494,0.212679,0.363716


Wir prüfen, was hinter den Nullen steckt. Geschwindigkeit == 0 gehört zu Steh-Meldungen wie Anhalten oder Zündung an/aus — kein Defekt, sondern genau das Stillstands-Signal, das wir für die Standzeiten brauchen, also keine Anpassung. KM-Stand == 0 verteilt sich dagegen auf beide Meldungsquellen und lässt sich nicht erklären; hier passen wir bewusst nichts an und halten den Punkt offen, statt etwas zu reparieren, das wir nicht verstehen.

In [11]:
# Geschwindigkeit == 0: welche Meldungstypen stecken dahinter?
display(emr_raw.loc[emr_raw["Geschwindigkeit"] == 0, "Formularbeschreibung"].value_counts().head())
# KM-Stand == 0: hängt das an einer Meldungsquelle?
display(emr_raw.loc[emr_raw["KM-Stand"] == 0, "Meldungsquelle"].value_counts())

Formularbeschreibung
Anhalten                 22760
Place wurde aktiviert     8118
Ende Rollen               2765
Zündung an                2434
Zündung aus               2422
Name: count, dtype: int64

Meldungsquelle
SPEDION App    16066
TruckBox       15264
Name: count, dtype: int64

Wir prüfen, was hinter den Nullen steckt. Geschwindigkeit == 0 gehört zu Steh-Meldungen wie Anhalten oder Zündung an/aus — kein Defekt, sondern genau das Stillstands-Signal, das wir für die Standzeiten brauchen, also keine Anpassung. KM-Stand == 0 verteilt sich dagegen auf beide Meldungsquellen und lässt sich nicht erklären; hier passen wir bewusst nichts an und halten den Punkt offen, statt etwas zu reparieren, das wir nicht verstehen.

In [12]:
df_istdaten_emr = emr_raw.drop(columns=["Meldungsquelle", "QUELLE_DATEI"])
print(df_istdaten_emr.shape)
df_istdaten_emr.head()

(150967, 10)


,Fahrername,Formularnummer,Formularbeschreibung,Meldungszeit,Geschwindigkeit,KM-Stand,Breitengrad,Längengrad,Position,QUELLE_KENNZEICHEN
2678,NaN,30012,Reboot - Generic Third Party Telematic Device Position M...,2026-03-02 00:22:52,0,314612.18,53.66768,9.99212,"DE - 22419 Hamburg (Langenhorn), Essener Straße 4a",OD-TS 8041
2679,NaN,30006,Startup - Generic Third Party Telematic Device Position ...,2026-03-02 00:24:00,0,314612.18,53.66768,9.99212,"DE - 22419 Hamburg (Langenhorn), Essener Straße 4a",OD-TS 8041
2680,NaN,1615,Tägliche Meldung,2026-03-02 01:00:00,0,314612.18,53.66759,9.99222,"DE - 22419 Hamburg (Langenhorn), Essener Straße 4a",OD-TS 8041
2681,NaN,30012,Reboot - Generic Third Party Telematic Device Position M...,2026-03-02 01:22:28,0,314612.18,53.66760,9.99222,"DE - 22419 Hamburg (Langenhorn), Essener Straße 4a",OD-TS 8041
2682,NaN,30006,Startup - Generic Third Party Telematic Device Position ...,2026-03-02 01:23:32,0,314612.18,53.66760,9.99222,"DE - 22419 Hamburg (Langenhorn), Essener Straße 4a",OD-TS 8041


## Kennzeichen auf das Soll-Format bringen

Wir wenden dieselbe Normalisierung wie beim Soll an: Großschreibung, Umlaute zusammengezogen, Buchstaben- und Zifferngruppen mit Bindestrich. Danach stimmen elf der zwölf Fahrzeuge zeichengleich mit dem Soll überein. Übrig bleibt auf jeder Seite genau ein Fahrzeug ohne Partner: im Soll PLO-TS-859, in EMR TS-859. Da die anderen elf eindeutig zugeordnet sind, kann TS-859 nur dieses Soll-Fahrzeug sein; im EMR-Sheetnamen fehlt schlicht das Präfix, das die Schwesterfahrzeuge Plö-TS853, Plö-TS856 und Plö-TS857 tragen. Wir setzen das Präfix daher gezielt und machen aus TS-859 das Soll-Fahrzeug PLO-TS-859. Das Soll-Fahrzeug RW-435 hat in EMR kein Gegenstück, weil es dort keine Ist-Daten gibt.

In [13]:
umlaut_collapse = str.maketrans({"Ä": "A", "Ö": "O", "Ü": "U"})

def normalisiere_kennzeichen(wert):
    if pd.isna(wert):
        return wert
    wert = wert.upper().translate(umlaut_collapse)
    return "-".join(re.findall(r"[A-Z]+|[0-9]+", wert))

df_istdaten_emr["QUELLE_KENNZEICHEN"] = df_istdaten_emr["QUELLE_KENNZEICHEN"].map(normalisiere_kennzeichen)

# TS-859 ist das einzige EMR-Fahrzeug ohne Soll-Partner und PLO-TS-859 das einzige Soll-Fahrzeug
# ohne EMR-Partner -> selbes Fahrzeug, im Sheetnamen fehlte das Praefix
df_istdaten_emr["QUELLE_KENNZEICHEN"] = df_istdaten_emr["QUELLE_KENNZEICHEN"].replace({"TS-859": "PLO-TS-859"})

print(sorted(df_istdaten_emr["QUELLE_KENNZEICHEN"].unique()))

['OD-TS-8041', 'OD-TS-8044', 'OD-TS-8046', 'OD-TS-8048', 'OD-TS-8050', 'PI-EN-444', 'PI-EN-900', 'PLO-TS-853', 'PLO-TS-856', 'PLO-TS-857', 'PLO-TS-859', 'WL-PL-431']


## Vertiefte Datenprüfung

Vor dem Abschluss prüfen wir auf Dubletten - das hatten wir bei den Soll-Daten getan, hier bisher nicht. Kilometerstand, Koordinaten und die Position-Spalte haben wir bereits in den Schritten zuvor angesehen, sie brauchen hier keine erneute Prüfung.

In [14]:
print("Exakte Voll-Dubletten:", df_istdaten_emr.duplicated().sum())
df_istdaten_emr[df_istdaten_emr.duplicated(keep=False)] \
    .sort_values(["QUELLE_KENNZEICHEN", "Meldungszeit"]) \
    [["QUELLE_KENNZEICHEN", "Meldungszeit", "Formularbeschreibung", "Geschwindigkeit", "KM-Stand"]].head(10)

Exakte Voll-Dubletten: 1848


,QUELLE_KENNZEICHEN,Meldungszeit,Formularbeschreibung,Geschwindigkeit,KM-Stand
2735,OD-TS-8041,2026-03-02 06:33:07,Place wurde aktiviert,0,314612.665
2736,OD-TS-8041,2026-03-02 06:33:07,Place wurde aktiviert,0,314612.665
2738,OD-TS-8041,2026-03-02 06:33:09,Place wurde aktiviert,0,314612.665
2739,OD-TS-8041,2026-03-02 06:33:09,Place wurde aktiviert,0,314612.665
2784,OD-TS-8041,2026-03-02 06:54:31,Place wurde aktiviert,0,314621.130
2786,OD-TS-8041,2026-03-02 06:54:31,Place wurde aktiviert,0,314621.130
2788,OD-TS-8041,2026-03-02 06:54:32,Place wurde aktiviert,0,314621.130
2789,OD-TS-8041,2026-03-02 06:54:32,Place wurde aktiviert,0,314621.130
2791,OD-TS-8041,2026-03-02 06:54:34,Place wurde aktiviert,0,314621.130
2792,OD-TS-8041,2026-03-02 06:54:34,Place wurde aktiviert,0,314621.130


Es zeigen sich 1.848 exakte Voll-Dubletten — etwa dieselbe Place wurde aktiviert-Meldung doppelt zur exakt selben Sekunde, mit identischen Koordinaten und KM-Stand. Das sind echte Doppelmeldungen, keine zufällig gleichen Zeilen. Wir entfernen die exakten Dubletten, damit sie spätere Zählungen und Standzeit-Aggregationen nicht verfälschen.

In [15]:
vorher = len(df_istdaten_emr)
df_istdaten_emr = df_istdaten_emr.drop_duplicates().copy()
print(f"Exakte Dubletten entfernt: {vorher - len(df_istdaten_emr)} | verbleibend: {len(df_istdaten_emr)}")

Exakte Dubletten entfernt: 1848 | verbleibend: 149119


## Ist-Basis (EMR) abschließen und exportieren

Die bereinigte EMR-Basis wird in zwei Formaten gespeichert. Das Pickle ist das Arbeitsformat für die Folge-Notebooks: Es speichert die Datentypen mit, sodass die Meldungszeit datetime bleibt und die Koordinaten float bleiben, und beim erneuten Einlesen jedes parse_dates entfällt. Die CSV ist die menschenlesbare Beleg-Kopie für den Anhang. Bewusst behalten wird auch die Spalte Position, die von SPEDION rückwärts-geokodierte Klartext-Adresse je Meldung. PRC liefert nur rohe Koordinaten, daher ist diese Adresse eine EMR-eigene Zusatzinfo, die später beim Einordnen von Standzeiten und ungeplanten Stopps hilft.

In [16]:
# Gemeinsame Spaltennamen mit den Soll-Daten (für den späteren Join)
df_istdaten_emr = df_istdaten_emr.rename(columns={
    "QUELLE_KENNZEICHEN": "LKW_KENNZ",
    "Fahrername": "FAHRERNAME",
})

# Bereinigte Ist-Basis festschreiben
df_istdaten_emr_clean = df_istdaten_emr.copy()

OUT_PATH = Path.cwd().parent / "data" / "processed"
OUT_PATH.mkdir(parents=True, exist_ok=True)

ziel_pkl = OUT_PATH / "istdaten_emr_clean.pkl"
ziel_csv = OUT_PATH / "istdaten_emr_clean.csv"

# Pickle = Arbeitsformat, CSV = lesbare Beleg-Kopie
df_istdaten_emr_clean.to_pickle(ziel_pkl)
df_istdaten_emr_clean.to_csv(ziel_csv, index=False, encoding="utf-8-sig")

print("Gespeichert:")
print("  Pickle:", ziel_pkl)
print("  CSV:   ", ziel_csv)
print("Form:", df_istdaten_emr_clean.shape)
print("Spalten:", df_istdaten_emr_clean.columns.tolist())
print()

# Pickle einlesen und Datentypen prüfen
_check = pd.read_pickle(ziel_pkl)
print("Typen-Kontrolle nach Round-Trip:")
for _sp in ["Meldungszeit", "Breitengrad", "Längengrad", "Geschwindigkeit", "Position"]:
    print(f"  {_sp:<14} -> {_check[_sp].dtype}")

Gespeichert:
  Pickle: c:\Users\jonas\OneDrive\Dokumente\FH_Kiel\VS_Code\Python_Projekte\hts_cargo_project\data\processed\istdaten_emr_clean.pkl
  CSV:    c:\Users\jonas\OneDrive\Dokumente\FH_Kiel\VS_Code\Python_Projekte\hts_cargo_project\data\processed\istdaten_emr_clean.csv
Form: (149119, 10)
Spalten: ['FAHRERNAME', 'Formularnummer', 'Formularbeschreibung', 'Meldungszeit', 'Geschwindigkeit', 'KM-Stand', 'Breitengrad', 'Längengrad', 'Position', 'LKW_KENNZ']

Typen-Kontrolle nach Round-Trip:
  Meldungszeit   -> datetime64[us]
  Breitengrad    -> float64
  Längengrad     -> float64
  Geschwindigkeit -> int64
  Position       -> str
